# Hausse de loyer Équinoxe : notebook de départ

**Votre tâche : estimer la hausse de loyer d'Équinoxe pour 2026.**

Ce notebook vous aide à vous orienter et vous montre une méthode correcte. Les prévisions, les données externes et les décisions de jugement
vous reviennent. Si c'est la première fois que vous utilisez un notebook, pas d'inquiétude. C'est un outil très simple à utiliser et puissant, qui rend les flux de travail clairs et faciles à communiquer. Il est couramment utilisé
en apprentissage profond, et le maîtriser vous aidera énormément si vous vous dirigez un jour vers la science des données ou tout autre domaine analytique. **CTRL/COMMAND + MAJ + H** affiche les raccourcis. Apprenez à supprimer, restaurer, créer et exécuter des cellules. Ça vous aidera.

Lisez les cellules, ne vous contentez pas de les exécuter.

#### Important : vous verrez que la description des données ne correspond pas toujours parfaitement à ce qu'elles sont réellement. Dans ce cas-ci, PromoPay dans les concessions est décrit comme un frais mensuel, ce qui n'est pas du tout le cas. C'est assez facile à comprendre : si la promo égalait le loyer, le résident habiterait en pratique gratuitement.
##### Votre travail consiste aussi à découvrir ce que les données représentent réellement, et à les renommer selon votre propre description. Il n'y a pas de réponse unique.

## 1. Charger les données

## 1b. Lire les colonnes : champs `h` et champs `s`

Chaque colonne de ce jeu commence par **`h`** ou **`s`**. C'est la convention de Yardi, et
se tromper là-dessus est la façon la plus courante de corrompre une requête de gestion immobilière.

| préfixe | signification | exemple | à quoi ça sert |
|---|---|---|---|
| **`h`** | **handle** (identifiant) - une clé numérique | `hUnit`, `hProperty`, `hBuilding` | **les jointures**. Opaque, stable, unique. |
| **`s`** | **stored value** (valeur stockée) - ce qu'un humain lit | `sUnitCode`, `sRent`, `sBuilding` | **l'affichage et le regroupement**. |

Yardi conserve les deux moitiés d'une même information côte à côte - `hAgent` à côté de `sAgent`,
`hProperty` à côté de `sPropCode`. Elles ne sont **pas interchangeables** :

> **Joindre sur `h`. Étiqueter avec `s`.**

Ce n'est pas du pinaillage, c'est tout l'objet de la section 4 ci-dessous. `sUnitCode` vaut `"0203"` - et
trois appartements différents sur un même site portent cette même chaîne. `hUnit` est unique par
construction. Si vous associez une unité à elle-même du côté `s`, vous fusionnez des appartements sans le savoir;
si vous associez sur `hUnit`, c'est impossible.

**Deux mises en garde sur `s`, qui reflètent toutes deux le vrai comportement de Yardi, pas des particularités de ce jeu :**

1. **`s` ne veut pas dire « string ».** La table `UNIT` de Yardi stocke `SRENT` et `DSQFT` comme
   *valeurs numériques*. Ici, `sRent` et `sSqft` sont des nombres. Ne déduisez pas un type d'un préfixe -
   vérifiez-le.
2. **Le vrai Yardi subdivise davantage le côté `s`** - `dt*` pour les dates (`dtLeaseFrom`), `d*`
   pour les doubles (`dSqft`), `i*` pour les entiers, `b*` pour les indicateurs. Ce jeu les ramène tous
   à `s` pour garder l'attention sur la distinction `h` / `s`. Quand vous ouvrirez la vraie base de données, attendez-vous
   à un alphabet plus long.

### Les codes de frais

`sChargeCode` dans le fichier des concessions correspond au plan comptable de Yardi. Les codes semblent abîmés
parce que `sCode` est un `nchar(8)` - « Free other » devient `Freeothe`, « Referencing » devient
`Referenc`. Huit caractères, tronqués, avec une casse incohérente. C'est à ça que ressemblent les vrais.

| `sChargeCode` | `sChargeDesc` | ce que c'est réellement |
|---|---|---|
| `PromoPay` | Promo - Payable in the future | Un rabais promotionnel. **Lisez la mise en garde ci-dessous.** |
| `freerent` | Free rent | Loyer annulé pour une partie du bail. |
| `freepark` | Free Parking | Frais de stationnement annulés. Un avantage *accessoire*, pas du loyer. |
| `freelock` | Free Locker | Casier de rangement offert. Accessoire. |
| `freeappl` | Free Appliances | Frais d'électroménagers annulés. Accessoire. |
| `Freeothe` | Free other | Tout ce qui n'est pas catégorisé. À traiter avec méfiance. |
| `Indem` | Client indemnity | Un paiement versé au résident, par exemple pour régler un différend. |
| `Referenc` | Referencing program | Un crédit de référencement. |

**Celui qu'il faut examiner avant de s'y fier :** `PromoPay` est décrit comme un frais
mensuel. Ce n'en est pas un. Il est très majoritairement inscrit sur un seul mois, pour un montant proche d'un mois
de loyer complet. Prenez la description au pied de la lettre et vous conclurez que les résidents habitent
presque gratuitement. Comparez `sMonths` et la taille de `sAmount` à `sRent` avant de le modéliser -
et décidez vous-même comment l'étaler sur la durée du bail.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

listings   = pd.read_csv("equinoxe_listings.csv")
leases     = pd.read_csv("equinoxe_lease_history.csv")
concessions= pd.read_csv("equinoxe_concessions.csv")
asking     = pd.read_csv("equinoxe_asking_history.csv")

for name, df in [("listings", listings), ("leases", leases),
                 ("concessions", concessions), ("asking", asking)]:
    print(f"{name:<12} {len(df):>6,} lignes   {df.shape[1]:>2} colonnes")

In [ ]:
# Convertir les dates une seule fois, dès le départ.
for col in ["sLeaseFrom", "sLeaseTo", "sSignDate", "sAvailable"]:
    if col in leases.columns:
        leases[col] = pd.to_datetime(leases[col], errors="coerce")

leases["year"] = leases.sLeaseFrom.dt.year
asking["date"] = pd.to_datetime(asking.sMonth + "-01")

leases.head(3)

## 2. Ce que vous avez sous les yeux

Six immeubles répartis entre Laval, Mont-Royal, Ottawa et Pointe-Claire. Tout est en date du **2025-12-31**;
il n'y a pas de 2026 dans ce jeu, parce que 2026 est justement ce que vous prédisez.

Cinq sont au Québec et un, The Met, est en Ontario. Ce n'est pas
un détail : les deux provinces encadrent les hausses de loyer selon des régimes différents, et
les traiter comme un seul portefeuille est un choix de modélisation que vous devrez défendre.

**Une chose à bien comprendre avant de commencer.** Il s'agit d'un *extrait publié*, pas du
portefeuille. Il contient une part fixe des unités de chaque immeuble et de chaque type de chambre - la
répartition est représentative, les volumes ne le sont pas. Donc tout ce qui nécessite un dénominateur d'unités
réelles est hors de portée ici : **l'occupation, l'inoccupation, l'absorption et le rôle de loyers total du portefeuille
ne peuvent pas être calculés à partir de ce jeu, et un chiffre que vous en tireriez serait un artefact de
l'extrait plutôt qu'un fait sur l'entreprise.** C'est la *variation* des loyers que ces données permettent de mesurer,
et c'est la variation des loyers qu'on vous demande.

In [ ]:
print(listings.groupby(["sState", "sCity", "sBuilding"]).size().to_string())
print()
print(listings.groupby(["sBeds", "sBaths"])
      .agg(units=("sRent", "size"),
           median_rent=("sRent", "median"),
           median_sqft=("sSqft", "median"))
      .to_string())

## 3. L'approche évidente

Le premier réflexe naturel : le loyer médian par année.

In [ ]:
naive = (leases[leases.year.between(2019, 2025)]
         .groupby("year")
         .agg(leases=("sRent", "size"), median_rent=("sRent", "median")))
naive["yoy_pct"] = (naive.median_rent.pct_change() * 100).round(2)
naive

Selon ce tableau, les loyers auraient augmenté de **10,8 % en 2023**. Ce n'est pas le cas.

La médiane bouge dès que la *composition* bouge, et la composition a beaucoup bougé. Plusieurs
immeubles ont été mis en service en 2023 et 2024. Regardez ce qui se passe sous la
médiane :

In [ ]:
window = leases[leases.year.between(2019, 2025)]
mix = window.groupby("year").agg(
    leases=("sRent", "size"),
    median_sqft=("sSqft", "median"),
    pct_2bed_plus=("sBeds", lambda s: (s >= 2).mean() * 100),
    buildings=("sBuilding", "nunique"))
mix["rent_per_sqft"] = (window.assign(psf=window.sRent / window.sSqft)
                        .groupby("year").psf.median())
mix.round(2)

In [ ]:
# À partir de quand chaque sBuilding a-t-il commencé à générer des baux?
print(window.groupby("sBuilding").year.min().sort_values().to_string())
print()
# Et combien demandent-ils? (baux de 2024)
print(window[window.year == 2024].groupby("sBuilding")
      .agg(leases=("sRent", "size"), median_rent=("sRent", "median"))
      .sort_values("median_rent").to_string())

Les unités louées en 2024 sont **plus petites** que celles louées en 2021 : la superficie
médiane est passée de 1 175 à 1 045 pi², et pourtant le loyer médian a fortement augmenté. C'est
le contraire de ce que la taille seule laisserait prévoir. La part des baux de 2 chambres et plus
est passée de 78 % à 62 % sur la même période.

Ce qui s'est réellement passé : Le Carlyle, The Met et Westpark ont été mis en service en 2023 et 2024. Ce sont des
immeubles haut de gamme, qui louent des unités plus petites, davantage axées sur les 1 chambre, à des tarifs au pied carré
bien plus élevés - la médiane 2024 de The Met est de 2 752 $, contre 1 672 $ pour Lévesque.
Le loyer au pied carré est passé de 1,71 $ à 2,20 $ en deux ans, et presque rien de
cela ne vient d'un locataire existant qui paie plus.


## 4. La méthode qui fonctionne : comparer une unité à elle-même

Associez les baux consécutifs de chaque unité et annualisez la variation. Même appartement,
même immeuble, tout pareil : ce qui reste, c'est le prix.

**La seule chose à ne pas rater :** associez sur `sPropCode` + `sUnitCode`.
Pas sur `sSite` + `sUnitCode`. Plusieurs sites regroupent plus d'un code de propriété avec
des numéros d'unité qui se chevauchent, et **248 des 1 061 unités entrent en collision** si vous utilisez
le site comme clé. Exécutez la cellule suivante pour le constater, puis utilisez `sPropCode` + `sUnitCode`, ou
la colonne `sLink`, qui est construite sur la clé sûre.

In [ ]:
# Pourquoi sSite + sUnitCode n'est pas une clé : un site, un numéro d'unité, plusieurs appartements.
# (hUnit est l'identifiant qui EST unique - voir le dictionnaire de données plus haut.)
dupes = listings[listings.duplicated(["sSite", "sUnitCode"], keep=False)]
print(f"{len(dupes):,} unités sur {len(listings):,} entrent en collision sur sSite + sUnitCode")
print()
print(listings[(listings.sSite == "saint-elzear") & (listings.sUnitCode == "0203")]
      [["hUnit", "sSite", "sPropCode", "sUnitCode", "sBeds", "sBaths", "sSqft", "sRent"]]
      .to_string(index=False))

In [ ]:
UNIT_KEY = ["sPropCode", "sUnitCode"]      # <-- pas ["sSite", "sUnitCode"]

def same_unit_growth(df, price_col="sRent", date_col="sLeaseFrom",
                     min_gap=0.5, max_gap=2.5):
    """Variation annualisée du loyer, en comparant chaque unité à son propre bail précédent.

    Conserve les écarts entre `min_gap` et `max_gap` années : plus court signifie
    généralement le même bail enregistré deux fois, plus long est trop ancien pour une comparaison équivalente.
    """
    d = df.sort_values(UNIT_KEY + [date_col]).copy()
    d["prev_price"] = d.groupby(UNIT_KEY)[price_col].shift(1)
    d["prev_date"]  = d.groupby(UNIT_KEY)[date_col].shift(1)
    d = d.dropna(subset=["prev_price", "prev_date"])

    d["gap_years"] = (d[date_col] - d.prev_date).dt.days / 365.25
    d = d[d.gap_years.between(min_gap, max_gap, inclusive="neither")]
    d = d[(d.prev_price > 0) & (d[price_col] > 0)]

    d["growth_pct"] = ((d[price_col] / d.prev_price) ** (1 / d.gap_years) - 1) * 100
    return d

pairs = same_unit_growth(leases)
print(f"{len(pairs):,} paires utilisables sur {pairs.groupby(UNIT_KEY).ngroups:,} unités")

honest = (pairs[pairs.year.between(2021, 2025)]
          .groupby("year")
          .growth_pct.agg(pairs="size", median="median").round(2))
honest

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
yrs = range(2021, 2026)
ax.plot(yrs, naive.loc[yrs, "yoy_pct"], "o--", label="naïf : médiane de tous les baux", lw=2)
ax.plot(yrs, honest.loc[yrs, "median"], "o-", label="même unité comparée à elle-même", lw=2)
ax.axhline(0, color="grey", lw=0.8)
ax.set_ylabel("croissance annuelle des loyers (%)")
ax.set_title("La méthode naïve mesure surtout la composition")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()



## 5. Loyer contractuel n'est pas loyer perçu

`sRent` est ce que le bail indique. `sRentEffective` est ce qu'Équinoxe a réellement
perçu après les mois gratuits, le stationnement gratuit et les casiers gratuits. Les concessions sont maintenant
la norme, et elles progressent.

In [ ]:
conc = (leases[leases.year.between(2021, 2025)]
        .groupby("year")
        .agg(leases=("sRent", "size"),
             pct_with_concession=("sConcession", lambda s: s.mean() * 100),
             median_contracted=("sRent", "median"),
             median_effective=("sRentEffective", "median")))
conc["gap_pct"] = ((conc.median_effective / conc.median_contracted - 1) * 100)
conc.round(2)

In [ ]:
# La concession change-t-elle l'évolution, ou seulement le niveau?
eff = same_unit_growth(leases, price_col="sRentEffective")
compare = pd.DataFrame({
    "contracted": honest["median"],
    "effective":  eff[eff.year.between(2021, 2025)].groupby("year").growth_pct.median(),
}).round(2)
compare

Ces deux colonnes évoluent de près jusqu'en 2023 - à moins
d'un point l'une de l'autre - puis elles divergent : en 2025, la croissance contractuelle est de
7,0 % alors que la croissance effective est de 3,6 %. L'écart de *niveau* du tableau précédent
se creuse tout au long de la période.

Elles répondent à des questions différentes, et l'année où elles se séparent est celle où les concessions
deviennent plus importantes plutôt que simplement plus fréquentes. Si vous pouvez expliquer ce que cette séparation signifie pour un
propriétaire - et laquelle des deux vous citeriez à un propriétaire - vous comprenez les données
mieux que la plupart des équipes.

## 6. Les renouvellements ne se comportent pas comme les relocations

`sRenewal = 1`, c'est un locataire en place qui renouvelle. `sRenewal = 0`, c'est une unité
qui se libère et qui est relouée au prix du marché. Au Québec, ces deux cas sont régis par
des règles différentes, et l'écart entre les deux est toute la raison pour laquelle un propriétaire s'intéresse
à ce chiffre. Remarquez que l'écart change de camp au cours de la période : les renouvellements mènent en 2022, les relocations en 2025.

In [ ]:
split = (pairs[pairs.year.between(2022, 2025)]
         .groupby(["year", "sRenewal"])
         .growth_pct.agg(pairs="size", median="median").round(2)
         .unstack("sRenewal"))
split.columns = [f"{a}_{'renewal' if b else 'turnover'}" for a, b in split.columns]
split

## 7. À vous de jouer

Tout ce qui précède est descriptif. Rien de tout ça n'est une prévision, et 2026 n'est pas dans
ce jeu. Ce qui reste, c'est le vrai travail :

**a) Décidez à quelle question vous répondez.** « La hausse de loyer de l'année », ce sont en fait
plusieurs chiffres différents : même unité contractuel, même unité effectif, renouvellements
seulement, relocations seulement, portefeuille combiné. Ils ne concordent pas. Choisissez-en un, nommez-le dans
votre rapport et justifiez votre choix.

**b) Intégrez des données publiques.**

- **Enquête sur les logements locatifs de la SCHL** : loyers et taux d'inoccupation par nombre de chambres pour les
  RMR de Montréal et d'Ottawa, ainsi que la répartition entre logements ayant changé de locataire ou non, qui correspond
  directement à `sRenewal`
- **IPC de Statistique Canada**, composante loyers, Québec et Ontario
- **Tribunal administratif du logement** : les paramètres annuels d'augmentation de loyer au Québec.
  Cinq des six immeubles y sont assujettis
- **Taux légal d'augmentation des loyers en Ontario** : s'applique à l'immeuble The Met, selon
  des règles différentes. N'appliquez pas le plafond d'une province à l'autre

Utilisez la série externe sur l'inoccupation comme *contexte* pour l'orientation des loyers. N'essayez pas
de la transposer en taux d'occupation propre à ce portefeuille - voir la note de la section 2.

**c) Faites une prévision.** Une extrapolation de tendance est un point de départ légitime. Une approche
qui concilie la tendance interne avec le marché externe est meilleure. Énoncez
vos hypothèses.

**e) Misez sur le raisonnement et faites de votre mieux.** Si votre méthode atteint une précision de 95 % et que quelqu'un d'autre obtient 96 %, ça ne veut pas dire que l'autre a gagné. Vous êtes évalué davantage sur la technique que sur le résultat. Les résultats comptent, mais sans un raisonnement constamment démontré derrière chaque choix, même 99 % ne veut rien dire. Nous devons voir une logique claire derrière vos décisions. Note : vous devez **TOUJOURS** expliquer une étape importante, même si ça vous semble inutile ou peu naturel.

In [ ]:
# Gabarit -- à compléter.

def estimate_2026(leases, asking, external=None):
    """Retourne votre estimation de la hausse de loyer 2026, en pourcentage.

    Indiquez dans votre rapport à quelle définition vous répondez et pourquoi.
    """
    raise NotImplementedError


def backtest(leases, target_year):
    """Exécute votre méthode comme si vous étiez à la fin de target_year - 1,
    et compare le résultat avec ce qui s'est réellement passé en target_year."""
    raise NotImplementedError


# Première vérification suggérée une fois estimate_2026 fonctionnel :
# for yr in (2023, 2024, 2025):
#     print(yr, backtest(leases, yr))

## Ce qui rapporte des points

La méthode avant la proximité du résultat. Une équipe qui énonce sa définition, associe les mêmes unités sur
la bonne clé, traite les concessions explicitement et tombe un peu à côté
obtiendra une meilleure note qu'une équipe qui tombe plus près en faisant la moyenne de quelque chose qu'elle n'a jamais
examiné.

Plus précisément, rapportent des points :

- Nommer la définition à laquelle vous répondez
- Comparer les mêmes unités plutôt que des médianes d'une année à l'autre
- Repérer et expliquer le problème de composition
- Utiliser `sRentEffective`, ou justifier pourquoi vous avez utilisé le loyer contractuel
- Séparer les renouvellements des relocations
- Concilier les données de marché externes avec la tendance interne
- Valider votre propre méthode sur l'historique (backtest)

## Ce qui fait perdre des points :

- Les nombres magiques et les variables mal nommées. Restez cohérent dans votre logique de nommage.
- Utiliser `sSite` + `sUnitCode` comme clé pour comparer les mêmes unités
- Une mauvaise utilisation des variables (features).
- Citer un taux d'occupation, d'inoccupation ou d'absorption tiré de cet extrait
- Aucune cellule markdown

### Ce qui donne des points bonus :

- Traiter l'immeuble The Met selon les règles de l'Ontario.